In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [4]:
merged_df = pd.read_csv("../data/final/merged_dataset.csv")
feedback_df = pd.read_csv("../data/final/feedback_dataset.csv")

print("Dataset lama :", len(merged_df))
print("Feedback     :", len(feedback_df))

Dataset lama : 3932
Feedback     : 34


In [5]:
feedback_df.columns = feedback_df.columns.str.strip()

feedback_df["is_correct"] = (
    feedback_df["is_correct"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

print(feedback_df.columns.tolist())
print(feedback_df["is_correct"].value_counts(dropna=False))

['text', 'predicted_label', 'corrected_label', 'is_correct']
is_correct
True     29
False     5
Name: count, dtype: int64


In [6]:
wrong_feedback = feedback_df[
    feedback_df["is_correct"] == False
].copy()

print("Feedback salah:", len(wrong_feedback))

display(
    wrong_feedback[
        ["text", "predicted_label", "corrected_label"]
    ]
)

Feedback salah: 5


,text,predicted_label,corrected_label
18,Biaya konsultasi dokter,healthcare,healthcare
20,Bayar uang sekolah adik,shopping,education
21,Beli kopi di kedai,shopping,food
32,Beli kopi di kedai,shopping,food
33,Bayar uang sekolah adik,shopping,education


In [7]:
feedback_to_add = wrong_feedback[
    ["text", "corrected_label"]
].copy()

feedback_to_add = feedback_to_add.rename(
    columns={
        "corrected_label": "label"
    }
)

print("Feedback siap ditambahkan:", len(feedback_to_add))
display(feedback_to_add)

Feedback siap ditambahkan: 5


,text,label
18,Biaya konsultasi dokter,healthcare
20,Bayar uang sekolah adik,education
21,Beli kopi di kedai,food
32,Beli kopi di kedai,food
33,Bayar uang sekolah adik,education


In [8]:
print(merged_df.columns.tolist())
display(merged_df.head())

['text', 'label']


,text,label
0,Pembayaran restoran sebesar Rp 48698 via UPI R...,food
1,Pembelian tiket kereta sebesar Rp 9244 via UPI...,travel
2,Pembelian paket makan siang sebesar Rp 5797 vi...,food
3,SIP reksa dana sebesar Rp 2052 via UPI Ref 198246,investment
4,Pembelian tiket kereta sebesar Rp 33218 via UP...,travel


In [9]:
existing_text = set(
    merged_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
)

feedback_to_add = feedback_to_add[
    ~feedback_to_add["text"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(existing_text)
].copy()

print("Feedback baru:", len(feedback_to_add))
display(feedback_to_add)

Feedback baru: 0


,text,label


In [10]:
new_dataset = pd.concat(
    [
        merged_df,
        feedback_to_add
    ],
    ignore_index=True
)

print("Dataset lama :", len(merged_df))
print("Feedback baru:", len(feedback_to_add))
print("Dataset baru :", len(new_dataset))

Dataset lama : 3932
Feedback baru: 0
Dataset baru : 3932


In [11]:
new_dataset.to_csv(
    "../data/final/merged_dataset_updated.csv",
    index=False
)

print("Dataset updated berhasil disimpan.")

Dataset updated berhasil disimpan.


In [12]:
print("Distribusi dataset lama:")
print(merged_df["label"].value_counts())

print("\nDistribusi dataset baru:")
print(new_dataset["label"].value_counts())

Distribusi dataset lama:
label
education        517
healthcare       453
transfer         355
shopping         324
bills            322
entertainment    248
food             226
topup            226
transport        220
donation         219
income           198
loan             184
fees             161
travel           149
investment       130
Name: count, dtype: int64

Distribusi dataset baru:
label
education        517
healthcare       453
transfer         355
shopping         324
bills            322
entertainment    248
food             226
topup            226
transport        220
donation         219
income           198
loan             184
fees             161
travel           149
investment       130
Name: count, dtype: int64


In [13]:
feedback_to_add = wrong_feedback[
    ["text", "corrected_label"]
].copy()

feedback_to_add = feedback_to_add.rename(
    columns={
        "corrected_label": "label"
    }
)

print("Feedback salah:")
display(feedback_to_add)

Feedback salah:


,text,label
18,Biaya konsultasi dokter,healthcare
20,Bayar uang sekolah adik,education
21,Beli kopi di kedai,food
32,Beli kopi di kedai,food
33,Bayar uang sekolah adik,education


In [14]:
updated_dataset = merged_df.copy()

for _, row in feedback_to_add.iterrows():

    mask = (
        updated_dataset["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        str(row["text"])
        .strip()
        .lower()
    )

    updated_dataset.loc[mask, "label"] = row["label"]

In [15]:
print("Dataset lama :", len(merged_df))
print("Dataset baru :", len(updated_dataset))

Dataset lama : 3932
Dataset baru : 3932


In [16]:
for _, row in feedback_to_add.iterrows():

    result = updated_dataset[
        updated_dataset["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        str(row["text"])
        .strip()
        .lower()
    ]

    print("\nTEXT:", row["text"])
    print("LABEL SETELAH UPDATE:", result["label"].tolist())


TEXT: Biaya konsultasi dokter
LABEL SETELAH UPDATE: ['healthcare']

TEXT: Bayar uang sekolah adik
LABEL SETELAH UPDATE: ['education']

TEXT: Beli kopi di kedai
LABEL SETELAH UPDATE: ['food']

TEXT: Beli kopi di kedai
LABEL SETELAH UPDATE: ['food']

TEXT: Bayar uang sekolah adik
LABEL SETELAH UPDATE: ['education']


In [19]:
check_texts = [
    "Biaya konsultasi dokter",
    "Bayar uang sekolah adik",
    "Beli kopi di kedai"
]

for text in check_texts:

    result = merged_df[
        merged_df["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        text.strip().lower()
    ]

    print("\nTEXT:", text)
    print(result[["text", "label"]].to_string(index=False))


TEXT: Biaya konsultasi dokter
                   text      label
Biaya konsultasi dokter healthcare

TEXT: Bayar uang sekolah adik
                   text     label
Bayar uang sekolah adik education

TEXT: Beli kopi di kedai
              text label
Beli kopi di kedai  food


In [20]:
for text in check_texts:

    old_labels = merged_df[
        merged_df["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        == text.strip().lower()
    ]["label"].tolist()

    new_labels = updated_dataset[
        updated_dataset["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        == text.strip().lower()
    ]["label"].tolist()

    print("\nTEXT:", text)
    print("SEBELUM :", old_labels)
    print("SESUDAH :", new_labels)


TEXT: Biaya konsultasi dokter
SEBELUM : ['healthcare']
SESUDAH : ['healthcare']

TEXT: Bayar uang sekolah adik
SEBELUM : ['education']
SESUDAH : ['education']

TEXT: Beli kopi di kedai
SEBELUM : ['food']
SESUDAH : ['food']


In [21]:
feedback_to_add = feedback_to_add.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

print("Feedback unik:", len(feedback_to_add))

display(feedback_to_add)

Feedback unik: 3


,text,label
0,Biaya konsultasi dokter,healthcare
1,Bayar uang sekolah adik,education
2,Beli kopi di kedai,food


In [22]:
for _, row in feedback_to_add.iterrows():

    existing = merged_df[
        merged_df["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        str(row["text"])
        .strip()
        .lower()
    ]

    old_labels = existing["label"].unique().tolist()

    print(
        f"{row['text']} | "
        f"Lama: {old_labels} | "
        f"Feedback: {row['label']}"
    )

Biaya konsultasi dokter | Lama: ['healthcare'] | Feedback: healthcare
Bayar uang sekolah adik | Lama: ['education'] | Feedback: education
Beli kopi di kedai | Lama: ['food'] | Feedback: food


In [23]:
changes = []

for _, row in feedback_to_add.iterrows():

    mask = (
        merged_df["text"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        str(row["text"])
        .strip()
        .lower()
    )

    old_labels = merged_df.loc[mask, "label"].unique().tolist()

    if len(old_labels) > 0 and row["label"] not in old_labels:
        changes.append({
            "text": row["text"],
            "old_label": old_labels,
            "new_label": row["label"]
        })

changes_df = pd.DataFrame(changes)

print("Jumlah perubahan:", len(changes_df))

display(changes_df)

Jumlah perubahan: 0


""


In [24]:
print(
    "Dataset lama:",
    len(merged_df)
)

print(
    "Dataset updated:",
    len(updated_dataset)
)

Dataset lama: 3932
Dataset updated: 3932


In [25]:
import pandas as pd

merged_df = pd.read_csv("../data/final/merged_dataset.csv")
feedback_df = pd.read_csv("../data/final/feedback_dataset.csv")

print("Dataset lama :", len(merged_df))
print("Feedback     :", len(feedback_df))

Dataset lama : 3932
Feedback     : 39


In [26]:
display(feedback_df.tail(10))

,text,predicted_label,corrected_label,is_correct
29,Bayar cicilan motor,loan,loan,True
30,Pesan hotel untuk liburan,travel,travel,True
31,Donasi ke panti asuhan,donation,donation,True
32,Beli kopi di kedai,shopping,food,False
33,Bayar uang sekolah adik,shopping,education,False
34,Bayar parkir kampus,shopping,transport,False
35,Beli kopi starling,shopping,food,False
36,Bayar biaya kuliah semester 5,shopping,education,False
37,Kirim uang ke ibu,shopping,transfer,False
38,Bayar premi asuransi prudentian,bills,healthcare,False


In [27]:
feedback_df["is_correct"] = (
    feedback_df["is_correct"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

wrong_feedback = feedback_df[
    feedback_df["is_correct"] == False
].copy()

print("Feedback salah:", len(wrong_feedback))

display(
    wrong_feedback[
        ["text", "predicted_label", "corrected_label"]
    ]
)

Feedback salah: 10


,text,predicted_label,corrected_label
18,Biaya konsultasi dokter,healthcare,healthcare
20,Bayar uang sekolah adik,shopping,education
21,Beli kopi di kedai,shopping,food
32,Beli kopi di kedai,shopping,food
33,Bayar uang sekolah adik,shopping,education
34,Bayar parkir kampus,shopping,transport
35,Beli kopi starling,shopping,food
36,Bayar biaya kuliah semester 5,shopping,education
37,Kirim uang ke ibu,shopping,transfer
38,Bayar premi asuransi prudentian,bills,healthcare


In [28]:
print(merged_df.columns)

Index(['text', 'label'], dtype='object')


In [29]:
new_data = wrong_feedback[
    ["text", "corrected_label"]
].copy()

new_data = new_data.rename(
    columns={
        "corrected_label": "label"
    }
)

display(new_data)

,text,label
18,Biaya konsultasi dokter,healthcare
20,Bayar uang sekolah adik,education
21,Beli kopi di kedai,food
32,Beli kopi di kedai,food
33,Bayar uang sekolah adik,education
34,Bayar parkir kampus,transport
35,Beli kopi starling,food
36,Bayar biaya kuliah semester 5,education
37,Kirim uang ke ibu,transfer
38,Bayar premi asuransi prudentian,healthcare


In [30]:
existing = set(
    zip(
        merged_df["text"].astype(str).str.strip().str.lower(),
        merged_df["label"].astype(str).str.strip().str.lower()
    )
)

new_rows = []

for _, row in new_data.iterrows():

    key = (
        str(row["text"]).strip().lower(),
        str(row["label"]).strip().lower()
    )

    if key not in existing:
        new_rows.append(row)

In [31]:
new_rows_df = pd.DataFrame(new_rows)

print("Data baru yang akan ditambahkan:", len(new_rows_df))

Data baru yang akan ditambahkan: 5


In [32]:
if not new_rows_df.empty:

    merged_df = pd.concat(
        [merged_df, new_rows_df],
        ignore_index=True
    )

print("Dataset setelah update:", len(merged_df))

Dataset setelah update: 3937


In [33]:
merged_df.to_csv(
    "../data/final/merged_dataset.csv",
    index=False
)

print("Dataset berhasil diperbarui.")

Dataset berhasil diperbarui.


In [34]:
df = pd.read_csv("../data/final/merged_dataset.csv")

X = df["text"]
y = df["label"]

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

In [38]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [39]:
svm = LinearSVC()

svm.fit(
    X_train_tfidf,
    y_train_encoded
)

LinearSVC()

In [40]:
y_pred = svm.predict(X_test_tfidf)

In [41]:
accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)

print("Accuracy:", accuracy)

print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=encoder.classes_
    )
)

Accuracy: 0.9631979695431472
               precision    recall  f1-score   support

        bills       0.97      0.98      0.98        64
     donation       1.00      1.00      1.00        44
    education       0.97      0.98      0.98       104
entertainment       1.00      0.98      0.99        50
         fees       1.00      1.00      1.00        32
         food       1.00      0.87      0.93        45
   healthcare       0.98      0.96      0.97        91
       income       1.00      0.88      0.93        40
   investment       1.00      1.00      1.00        26
         loan       1.00      0.97      0.99        37
     shopping       0.80      0.92      0.86        65
        topup       0.88      1.00      0.94        45
     transfer       0.99      0.97      0.98        71
    transport       1.00      0.98      0.99        44
       travel       1.00      0.97      0.98        30

     accuracy                           0.96       788
    macro avg       0.97      0.96

In [42]:
import joblib

joblib.dump(
    svm,
    "../models/model.pkl"
)

joblib.dump(
    vectorizer,
    "../models/vectorizer.pkl"
)

joblib.dump(
    encoder,
    "../models/label_encoder.pkl"
)

print("Model berhasil diperbarui.")

Model berhasil diperbarui.
